# ASL Model Training - Enhanced Version

This notebook trains the ASL classifier with:
- Enhanced early stopping (stops when accuracy drops after reaching ~99.98%)
- Model saved at each epoch
- Best model selection
- Comprehensive graph generation:
  1. Training Progress Graphs
  2. Confusion Matrix Visualization
  3. Per-Class Accuracy Bar Chart

All graphs are saved to the `GRAPHS/` folder.

## 1. Setup and Imports

In [1]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Check device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: mps
PyTorch version: 2.10.0


## 2. Load Dataset

In [2]:
from src.dataset import get_dataloaders, ASL_CLASSES

# Configuration
DATA_ROOT = 'data/asl_alphabet'
BATCH_SIZE = 64
VAL_SPLIT = 0.1

# Get dataloaders
train_loader, val_loader, test_loader = get_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    num_workers=4
)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Number of classes: {len(ASL_CLASSES)}")
print(f"Classes: {ASL_CLASSES}")

ValueError: No images found in data/asl_alphabet/asl_alphabet_train. Check your dataset path.

## 3. Create Model and Trainer

In [ ]:
from src.model import ASLClassifier
from src.train import Trainer

# Create model
model = ASLClassifier(
    num_classes=len(ASL_CLASSES),
    pretrained=True,
    dropout=0.2
)

# Create trainer with optimized hyperparameters
trainer = Trainer(
    model=model,
    device=device,
    learning_rate=1e-4,
    weight_decay=1e-4,
    label_smoothing=0.1
)

print("Model and Trainer created successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Train the Model

Training with enhanced early stopping:
- Monitors validation accuracy
- Stops if accuracy drops after reaching ~99.98%
- Saves model at each epoch
- Keeps track of the best model

In [ ]:
# Training configuration
EPOCHS = 50  # Max epochs
EARLY_STOPPING_PATIENCE = 5
HIGH_ACCURACY_THRESHOLD = 0.9998  # 99.98%
SAVE_DIR = 'models'

# Train the model
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    save_dir=SAVE_DIR,
    high_accuracy_threshold=HIGH_ACCURACY_THRESHOLD
)

## 5. Generate Comprehensive Graphs

This section generates all required visualizations and saves them to `GRAPHS/` folder.

In [ ]:
from src.train import generate_all_graphs

# Load the best model before evaluation
best_model_path = os.path.join(SAVE_DIR, 'asl_cnn_best.pt')
trainer.model.load_state_dict(torch.load(best_model_path))
print(f"Loaded best model from: {best_model_path}")

# Generate all graphs
cm, per_class_acc = generate_all_graphs(
    trainer=trainer,
    test_loader=test_loader,
    class_names=ASL_CLASSES,
    save_dir='GRAPHS'
)

## 6. Export Best Model to ONNX

In [ ]:
from src.model import export_to_onnx

# Export to ONNX format
onnx_path = os.path.join(SAVE_DIR, 'asl_cnn_best.onnx')
export_to_onnx(trainer.model, onnx_path)
print(f"Model exported to ONNX: {onnx_path}")

## 7. Training Summary

Display final metrics and statistics.

In [ ]:
import pandas as pd

# Create summary dataframe
summary_data = {
    'Metric': ['Training Accuracy', 'Validation Accuracy', 'Test Accuracy', 'Total Epochs'],
    'Value': [
        f"{max(history['train_acc'])*100:.4f}%",
        f"{max(history['val_acc'])*100:.4f}%",
        f"{(np.trace(cm) / np.sum(cm) * 100):.4f}%",
        len(history['train_loss'])
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

# Find best and worst performing classes
sorted_classes = sorted(per_class_acc.items(), key=lambda x: x[1])
print("\nTop 5 Best Performing Classes:")
for class_name, acc in sorted_classes[-5:][::-1]:
    print(f"  {class_name:15s}: {acc*100:.2f}%")

print("\nTop 5 Classes Needing Improvement:")
for class_name, acc in sorted_classes[:5]:
    print(f"  {class_name:15s}: {acc*100:.2f}%")

## 8. Visualize Sample Predictions (Optional)

In [ ]:
from src.dataset import denormalize
import torch.nn.functional as F

# Get a batch of test images
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

# Make predictions
trainer.model.eval()
with torch.no_grad():
    outputs = trainer.model(images)
    probs = F.softmax(outputs, dim=1)
    _, predicted = outputs.max(1)

# Visualize first 16 samples
fig, axes = plt.subplots(4, 4, figsize=(15, 15))
axes = axes.flatten()

for idx in range(min(16, len(images))):
    img = denormalize(images[idx]).cpu().permute(1, 2, 0).numpy()
    true_label = ASL_CLASSES[labels[idx]]
    pred_label = ASL_CLASSES[predicted[idx]]
    confidence = probs[idx][predicted[idx]].item() * 100
    
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    color = 'green' if true_label == pred_label else 'red'
    axes[idx].set_title(
        f"True: {true_label}\nPred: {pred_label}\nConf: {confidence:.1f}%",
        color=color,
        fontweight='bold'
    )

plt.tight_layout()
plt.savefig('GRAPHS/04_sample_predictions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Sample predictions saved to GRAPHS/04_sample_predictions.png")

## 9. Save Training History

In [ ]:
import json

# Save training history as JSON
history_data = {
    'train_loss': [float(x) for x in history['train_loss']],
    'train_acc': [float(x) for x in history['train_acc']],
    'val_loss': [float(x) for x in history['val_loss']],
    'val_acc': [float(x) for x in history['val_acc']],
    'per_class_accuracy': {k: float(v) for k, v in per_class_acc.items()},
    'overall_test_accuracy': float(np.trace(cm) / np.sum(cm)),
    'total_epochs': len(history['train_loss']),
    'best_val_accuracy': float(max(history['val_acc']))
}

history_path = os.path.join(SAVE_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history_data, f, indent=2)

print(f"Training history saved to: {history_path}")
print("\n✅ Training complete! All results saved.")